# Prédiction de conversion à la newsletter

**Objectif** : Prédire si un visiteur s'inscrit à la newsletter à partir de données de trafic. Métrique : F1 sur la classe « converti ».

**Source**
Ce projet a été réalisé dans le cadre de la formation en Data Science chez **Jedha Bootcamp**, qui a fourni le jeu de données de conversion. 

Dans le cadre de ce challenge (*Conversion Rate Challenge*), les données proviennent de l'historique de trafic et d'inscriptions de la célèbre newsletter hebdomadaire **[Data Science Weekly](https://www.datascienceweekly.org/)**. L'objectif simulé est de concevoir un modèle prédictif pour identifier si un visiteur va s'abonner à la newsletter en fonction de ses caractéristiques (pays, âge, source, pages visitées).

**Licence et Conditions d'utilisation**
- **Données** (`conversion_data_train.csv` et `conversion_data_test.csv`) : Le dataset est mis à disposition par l'école dans un but strictement pédagogique, d'apprentissage et non commercial.
- **Code et Analyse** : L'ensemble du code, des analyses exploratoires (EDA) et des modèles de Machine Learning présentés dans ce dépôt est partagé sous licence MIT (libre et open-source).

**Exécution** : installer les dépendances (`pip install -r requirements.txt`), puis lancer « Restart & Run All » depuis la racine du dépôt.


In [1]:
from pathlib import Path

FICHIERS_ATTENDUS = ['conversion_data_train.csv', 'conversion_data_test.csv']
manquants = [f for f in FICHIERS_ATTENDUS if not Path(f).exists()]
if manquants:
    raise FileNotFoundError(f"Fichiers manquants : {manquants}. Voir la section « Données » du README.")
print('Fichiers présents :', FICHIERS_ATTENDUS)


Fichiers présents : ['conversion_data_train.csv', 'conversion_data_test.csv']


# Conversion Rate Prediction

Projet : prédire si un visiteur de www.datascienceweekly.org s'inscrit à la newsletter, à partir de données de trafic.

- **Métrique** : F1-score sur la classe 1 (converti)
- **Données** : `conversion_data_train.csv` (labels disponibles), `conversion_data_test.csv` (sans labels, à prédire)
- **Livrable** : fichier de prédictions `conversion_data_test_predictions_antoine.csv`

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (f1_score, confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             precision_recall_curve)
from sklearn.inspection import permutation_importance

pio.templates.default = "plotly_white"

data = pd.read_csv('conversion_data_train.csv')
print('Set avec labels :', data.shape)
display(data.dtypes)

Set avec labels : (284580, 6)


country                  str
age                    int64
new_user               int64
source                   str
total_pages_visited    int64
converted              int64
dtype: object

# Aperçu des données

- `country` : pays de l'utilisateur (catégoriel)
- `age` : âge (numérique)
- `new_user` : 1 = nouveau visiteur, 0 = déjà vu
- `source` : origine du trafic (Ads, Direct, Seo)
- `total_pages_visited` : nombre de pages consultées
- `converted` : **cible** (1 = inscription à la newsletter, 0 = non)

In [3]:
def taux_par(col):
    """Taux de conversion moyen par modalité."""
    t = data.groupby(col, observed=True)['converted'].mean()
    return t.index.astype(str), t.values

data_sample = data.sample(10000, random_state=42)
tranches = pd.cut(data['age'], bins=range(10, 81, 10), right=False)

# ---- Bloc 1 : conversion, âge, pages, pays ----
fig1 = make_subplots(rows=2, cols=2, subplot_titles=(
    'Conversion', 'Âge', 'Pages visitées', 'Taux de conversion par pays'))

cc = data['converted'].value_counts().sort_index()
fig1.add_trace(go.Bar(x=['0 (non)', '1 (oui)'], y=cc.values), row=1, col=1)
fig1.add_trace(go.Histogram(x=data_sample['age'], nbinsx=50), row=1, col=2)
fig1.add_trace(go.Histogram(x=data_sample['total_pages_visited'], nbinsx=50), row=2, col=1)
x, y = taux_par('country')
fig1.add_trace(go.Bar(x=x, y=y), row=2, col=2)

fig1.update_layout(height=600, width=900, showlegend=False,
                   title_text="Analyse exploratoire - Bloc 1/2")
fig1.update_xaxes(title_text='Converti', row=1, col=1)
fig1.update_xaxes(title_text='Âge', row=1, col=2)
fig1.update_xaxes(title_text='Pages visitées', row=2, col=1)
fig1.update_xaxes(title_text='Pays', row=2, col=2)
fig1.update_yaxes(title_text='Count', row=1, col=1)
fig1.update_yaxes(title_text='Count', row=1, col=2)
fig1.update_yaxes(title_text='Count', row=2, col=1)
fig1.update_yaxes(title_text='Taux de conversion', row=2, col=2)
fig1.show()

# ---- Bloc 2 : source, nouveau visiteur, pages vs conversion, âge ----
fig2 = make_subplots(rows=2, cols=2, subplot_titles=(
    'Taux par source', 'Taux selon new_user', 'Pages vs conversion', 'Taux par tranche d\'âge'))

x, y = taux_par('source')
fig2.add_trace(go.Bar(x=x, y=y), row=1, col=1)
x, y = taux_par('new_user')
fig2.add_trace(go.Bar(x=x, y=y), row=1, col=2)
fig2.add_trace(go.Box(x=data_sample['converted'].astype(str),
                      y=data_sample['total_pages_visited']), row=2, col=1)
t_age = data.groupby(tranches, observed=True)['converted'].mean()
fig2.add_trace(go.Bar(x=t_age.index.astype(str), y=t_age.values), row=2, col=2)

fig2.update_layout(height=600, width=900, showlegend=False,
                   title_text="Analyse exploratoire - Bloc 2/2")
fig2.update_xaxes(title_text='Source', row=1, col=1)
fig2.update_xaxes(title_text='new_user', row=1, col=2)
fig2.update_xaxes(title_text='Converti', row=2, col=1)
fig2.update_xaxes(title_text='Tranche d\'âge', row=2, col=2)
fig2.update_yaxes(title_text='Taux de conversion', row=1, col=1)
fig2.update_yaxes(title_text='Taux de conversion', row=1, col=2)
fig2.update_yaxes(title_text='Pages visitées', row=2, col=1)
fig2.update_yaxes(title_text='Taux de conversion', row=2, col=2)
fig2.show()

In [4]:
print("Taux de conversion global : {:.2%}".format(data['converted'].mean()))
print("Visiteurs ayant consulté moins de 5 pages : {:.1%}".format((data['total_pages_visited'] < 5).mean()))

print("\nMédiane de pages visitées selon converted :")
print(data.groupby('converted')['total_pages_visited'].median())

print("\nTaux de conversion par pays :")
print(data.groupby('country')['converted'].mean().round(4))

print("\nTaux de conversion par source :")
print(data.groupby('source')['converted'].mean().round(4))

print("\nTaux de conversion selon new_user :")
print(data.groupby('new_user')['converted'].mean().round(4))

print("\nTaux de conversion par tranche d'âge :")
print(data.groupby(tranches, observed=True)['converted'].mean().round(4))

print("\nLignes avec âge >= 80 (hors tranches) :", (data['age'] >= 80).sum(),
      "| âge max :", data['age'].max())

Taux de conversion global : 3.23%
Visiteurs ayant consulté moins de 5 pages : 53.6%

Médiane de pages visitées selon converted :
converted
0     4.0
1    14.0
Name: total_pages_visited, dtype: float64

Taux de conversion par pays :
country
China      0.0013
Germany    0.0624
UK         0.0525
US         0.0379
Name: converted, dtype: float64

Taux de conversion par source :
source
Ads       0.0348
Direct    0.0278
Seo       0.0329
Name: converted, dtype: float64

Taux de conversion selon new_user :
new_user
0    0.0719
1    0.0141
Name: converted, dtype: float64

Taux de conversion par tranche d'âge :
age
[10, 20)    0.0644
[20, 30)    0.0427
[30, 40)    0.0222
[40, 50)    0.0120
[50, 60)    0.0049
[60, 70)    0.0105
[70, 80)    0.0000
Name: converted, dtype: float64

Lignes avec âge >= 80 (hors tranches) : 2 | âge max : 123


# Synthèse de l'analyse exploratoire

- **Taux de conversion** : 3,2 %, classes très déséquilibrées.
- **Pages visitées** : 53,6 % des visiteurs consultent moins de 5 pages. Médiane de 14 pages pour les convertis, contre 4 pour les autres.
- **Nouveaux visiteurs** : taux de conversion de 7,2 % pour les visiteurs déjà connus (`new_user` = 0), contre 1,4 % pour les nouveaux.
- **Pays** : écarts forts (Allemagne 6,2 %, Royaume-Uni 5,3 %, États-Unis 3,8 %, Chine 0,1 %). À vérifier avant interprétation (effectifs, qualité du tracking).
- **Source** : écarts faibles (Ads 3,5 %, Seo 3,3 %, Direct 2,8 %).
- **Âge** : conversion la plus forte dans la tranche 10-20 ans (6,4 %), puis 20-30 ans (4,3 %). Tranche à interpréter avec prudence. Deux lignes ont un âge ≥ 80 (maximum 123) : valeurs aberrantes, conservées dans le modèle.

In [5]:
features_list = ['country', 'age', 'new_user', 'source', 'total_pages_visited']
numeric_features = ['age', 'total_pages_visited', 'new_user']
categorical_features = ['country', 'source']
target = 'converted'

X = data[features_list]
Y = data[target]

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y)
print(f"Train : {X_train.shape}, Test : {X_test.shape}")

# Sortie dense : compatible avec tous les modèles, y compris HistGradientBoosting
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False),
     categorical_features)
])

Train : (227664, 5), Test : (56916, 5)


# Préparation

Split train/test 80/20, stratifié pour conserver la proportion de conversions dans chaque partie. Le test set est mis de côté et n'est utilisé qu'à l'étape d'évaluation finale.

In [6]:
baseline_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
baseline_model.fit(X_train, Y_train)
Y_pred_baseline = baseline_model.predict(X_test)

print("--- Baseline : régression logistique ---")
print(f"F1-score : {f1_score(Y_test, Y_pred_baseline):.4f}")
print("Matrice de confusion :\n", confusion_matrix(Y_test, Y_pred_baseline))

rf_baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])
rf_baseline.fit(X_train, Y_train)
Y_pred_rf_baseline = rf_baseline.predict(X_test)

print("\n--- Baseline : Random Forest (paramètres par défaut) ---")
print(f"F1-score : {f1_score(Y_test, Y_pred_rf_baseline):.4f}")
print("Matrice de confusion :\n", confusion_matrix(Y_test, Y_pred_rf_baseline))

--- Baseline : régression logistique ---
F1-score : 0.5122
Matrice de confusion :
 [[51870  3210]
 [   99  1737]]

--- Baseline : Random Forest (paramètres par défaut) ---
F1-score : 0.4988
Matrice de confusion :
 [[52073  3007]
 [  227  1609]]


# Comparaison des baselines

- **Régression logistique** : F1 = 0,512. Rappel élevé (95 %), mais précision faible (35 %) : le modèle signale beaucoup de faux positifs.
- **Random Forest** (paramètres par défaut) : F1 = 0,499 (rappel 87,6 %, précision 34,9 %). Sur ce critère, elle fait moins bien que la régression logistique.

La Random Forest reste le point de départ de l'optimisation : ses hyperparamètres peuvent améliorer le F1.

## Partie 1bis : Segmentation et réduction de dimension

Objectif : vérifier si les visiteurs se regroupent en profils naturels, et réduire la dimension des features encodées pour la visualisation.

**Attention** : Tout est fait **sur `X_train` uniquement** pour éviter tout data leakage.
Les clusters ne sont pas utilisés comme feature dans le modèle supervisé final ;
ils servent à l'analyse exploratoire et aux recommandations.

In [7]:
from sklearn.decomposition import PCA
# PCA / ACP : : Analyse en Composantes Principales pour réduction dimensionnalité

# On ajuste le préprocesseur sur le train uniquement
X_train_proc = preprocessor.fit_transform(X_train)
feature_names = preprocessor.get_feature_names_out()

# PCA : réduction de dimension pour visualisation
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_proc)

print("Variance expliquée :")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1} : {v:.3f}")
print(f"  Cumulé : {pca.explained_variance_ratio_.sum():.3f}")

# Contributions des features aux deux axes
pca_df = pd.DataFrame(
    pca.components_.T,
    index=feature_names,
    columns=["PC1", "PC2"]
)
display(pca_df.round(3))

# Nuage 2D coloré par conversion
pca_visu = pd.DataFrame(X_train_pca, columns=["PC1", "PC2"])
pca_visu["converted"] = Y_train.values

fig_pca = px.scatter(
    pca_visu.sample(5000, random_state=42),
    x="PC1", y="PC2", color=pca_visu["converted"].astype(str).sample(5000, random_state=42),
    title="Projection PCA des visiteurs (train) — coloré par conversion",
    opacity=0.5,
)
fig_pca.update_xaxes(title_text="PC1 — Engagement de navigation (pages+, nouveaux−)")
fig_pca.update_yaxes(title_text="PC2 — Âge (+0,877)")
fig_pca.show()

Variance expliquée :
  PC1 : 0.286
  PC2 : 0.258
  Cumulé : 0.544


,PC1,PC2
num__age,-0.386,0.877
num__total_pages_visited,0.687,0.068
num__new_user,-0.615,-0.475
cat__country_Germany,0.003,0.000
cat__country_UK,0.009,-0.001
cat__country_US,0.009,0.004
cat__source_Direct,-0.003,-0.001
cat__source_Seo,0.001,0.003


**Lecture du graphique PCA**

- **Rappel :** PC1 = axe d'engagement (pages+, nouveaux−), PC2 = axe d'âge (+0,877).
- **Les lignes diagonales** : elles proviennent de la nature discrète de `total_pages_visited` (nombres entiers). Chaque "rayure" correspond à un nombre de pages différent, projeté dans l'espace PC1/PC2.
- **La séparation des couleurs** est nette le long de **PC1** : les convertis (orange) se concentrent à droite (PC1 > 1,5), les non-convertis (bleu) à gauche. Cela confirme que **PC1 capture le signal de conversion**, porté par le nombre de pages visitées.
- **Sur l'axe PC2** (âge), les deux classes se recouvrent largement : l'âge seul ne sépare pas les convertis. Cohérent avec son importance plus faible dans le modèle.
- **Zone de recouvrement** (PC1 entre 1 et 2) : visiteurs modérément engagés, pour lesquels la prédiction reste difficile. C'est là que le modèle apporte de la valeur par rapport à une règle simple.
- **Point isolé en haut à gauche** (PC2 ≈ 9,5) : visiteur atypique (probablement un âge très élevé, cohérent avec les anomalies de saisie signalées en EDA).

In [8]:
# Choix de K : coude + silhouette sur un échantillon du train
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_sample = X_train_proc[:10000]  # échantillon pour accélérer

K_range = range(2, 8)
inertie, silhouettes = [], []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_sample)
    inertie.append(km.inertia_)
    silhouettes.append(silhouette_score(X_sample, km.labels_))

display(pd.DataFrame({'K': list(K_range), 'Inertie': inertie, 'Silhouette': silhouettes}).round(4))

,K,Inertie,Silhouette
0,2,28406.7861,0.3069
1,3,23643.7950,0.2311
2,4,19911.6438,0.2457
3,5,18020.3542,0.2347
4,6,16336.1707,0.2304
5,7,15316.4666,0.1951


In [9]:
# K-Means final sur tout le train
K_RETENU = 4
km_visiteurs = KMeans(n_clusters=K_RETENU, random_state=42, n_init=10).fit(X_train_proc)
clusters_train = km_visiteurs.labels_

# --- Profil de chaque cluster ---
X_train_profile = X_train.copy()
X_train_profile['cluster'] = clusters_train
X_train_profile['converted'] = Y_train.values

# Moyennes des features numériques + taux de conversion
profil = X_train_profile.groupby('cluster').agg(
    effectif=('converted', 'size'),
    taux_conversion=('converted', 'mean'),
    age_moyen=('age', 'mean'),
    pages_median=('total_pages_visited', 'median'),
    part_nouveaux=('new_user', 'mean'),
).round(3)
display(profil)

# Distribution des modalités dominantes
for col in ['country', 'source']:
    print(f"\n--- {col} (mode par cluster) ---")
    print(X_train_profile.groupby('cluster')[col].agg(lambda x: x.mode().iloc[0]))

,effectif,taux_conversion,age_moyen,pages_median,part_nouveaux
cluster,,,,,
0,83597,0.001,25.148,4.0,1.000
1,23359,0.288,27.819,11.0,0.620
2,58695,0.001,39.306,4.0,0.991
3,62013,0.007,30.614,4.0,0.000



--- country (mode par cluster) ---
cluster
0    US
1    US
2    US
3    US
Name: country, dtype: str

--- source (mode par cluster) ---
cluster
0    Seo
1    Seo
2    Seo
3    Seo
Name: source, dtype: str


## Lecture des segments

**PCA (54,4 % cumulé)** — PC1 (28,6 %) lit l'engagement de navigation (pages+, nouveaux−) ; PC2 (25,8 %) lit l'âge (+0,877). `country` et `source` ont une contribution négligeable, cohérent avec leur faible pouvoir discriminant en EDA.

**K=4** — silhouette maximale locale à 0,246 après K=3 (0,231). Compromis finesse / stabilité.

| Cluster | Effectif | Taux conv. | Pages méd. | % nouveaux | Lecture |
|---|---|---|---|---|---|
| 0 | 37 % | 0,1 % | 4 | 100 % | Nouveaux inactifs |
| **1** | **10 %** | **28,8 %** | **11** | 62 % | **Engagés / convertisseurs** |
| 2 | 26 % | 0,1 % | 4 | 99 % | Âgés inactifs |
| 3 | 27 % | 0,7 % | 4 | 0 % | Récurrents peu actifs |

**Enseignement** : le cluster 1 concentre ~92 % des conversions du train (taux 28,8 %, soit 9× la moyenne). Il se distingue uniquement par la navigation (11 pages médianes vs 4 pour les autres). Cela **confirme le rôle dominant de `total_pages_visited`** sans en faire une feature du modèle supervisé : le cluster contient de l'information issue de `Y_train` et créerait un risque de fuite. La segmentation sert de **lecture métier** du signal, pas de substitut au modèle.

# Partie 2 : Optimisation des modèles

La recherche d'hyperparamètres se fait sur un échantillon de 30 000 lignes du train, pour gagner du temps. Le test set n'intervient jamais dans cette étape.

In [10]:
sample_size = 30000
X_train_sample, _, Y_train_sample, _ = train_test_split(
    X_train, Y_train, train_size=sample_size, random_state=42, stratify=Y_train)
print(f"Recherche d'hyperparamètres sur {sample_size} lignes.")

logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
grid_log = GridSearchCV(logistic_pipeline, {'classifier__C': [0.1, 1, 10]},
                        cv=3, scoring='f1', n_jobs=-1)
grid_log.fit(X_train_sample, Y_train_sample)
print("Logistic : meilleurs paramètres", grid_log.best_params_,
      "| F1 validation =", round(grid_log.best_score_, 4))

Recherche d'hyperparamètres sur 30000 lignes.
Logistic : meilleurs paramètres {'classifier__C': 1} | F1 validation = 0.4952


In [11]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])
param_grid_rf = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [15, 30],
    'classifier__min_samples_split': [5, 10],
    'classifier__min_samples_leaf': [2, 4]
}
grid_rf = GridSearchCV(rf_pipeline, param_grid_rf, cv=3, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train_sample, Y_train_sample)
print("Random Forest : meilleurs paramètres", grid_rf.best_params_,
      "| F1 validation =", round(grid_rf.best_score_, 4))

Random Forest : meilleurs paramètres {'classifier__max_depth': 30, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100} | F1 validation = 0.5791


In [12]:
hgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(class_weight='balanced', random_state=42))
])
param_grid_hgb = {
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__max_depth': [3, 6],
    'classifier__max_iter': [200]
}
grid_hgb = GridSearchCV(hgb_pipeline, param_grid_hgb, cv=3, scoring='f1', n_jobs=-1)
grid_hgb.fit(X_train_sample, Y_train_sample)
print("Gradient boosting : meilleurs paramètres", grid_hgb.best_params_,
      "| F1 validation =", round(grid_hgb.best_score_, 4))

Gradient boosting : meilleurs paramètres {'classifier__learning_rate': 0.05, 'classifier__max_depth': 6, 'classifier__max_iter': 200} | F1 validation = 0.5277


# Note sur le F1 de validation

`best_score_` est la moyenne des F1 sur les folds de validation croisée, calculée sur l'échantillon de 30 000 lignes. Cette valeur sert uniquement à comparer les hyperparamètres entre eux.

# Sélection du modèle et du seuil de décision

Chaque candidat (Random Forest et gradient boosting, avec leurs meilleurs hyperparamètres) est évalué par validation croisée sur le **train complet**. Pour chacun, on cherche le seuil de décision qui maximise le F1. Le modèle retenu est celui dont le F1 de validation est le plus élevé. Le test set n'est pas utilisé dans cette étape.

Les F1 de la partie 2 (par exemple 0,579 pour le Random Forest) ne sont pas comparables à ceux de cette étape : ils sont calculés au seuil implicite de 0,5, sur 30 000 lignes. Ici, le seuil est optimisé sur le train complet. Au seuil 0,5, le gradient boosting était derrière (F1 de validation 0,528, contre 0,579 pour le Random Forest). Une fois le seuil optimisé, il passe devant (validation croisée sur le train : 0,766, contre 0,739).

In [13]:
def seuil_optimal(modele, X_, y_):
    """Seuil qui maximise le F1, calculé sur des prédictions out-of-fold du train."""
    proba = cross_val_predict(modele, X_, y_, cv=3, method='predict_proba', n_jobs=-1)[:, 1]
    precision, recall, seuils = precision_recall_curve(y_, proba)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    i = np.argmax(f1[:-1])  # le dernier point de la courbe n'a pas de seuil associé
    return seuils[i], f1[i]

candidats = {
    'Random Forest': clone(grid_rf.best_estimator_),
    'Gradient boosting': clone(grid_hgb.best_estimator_),
}

resultats = {}
for nom, modele in candidats.items():
    seuil, f1_cv = seuil_optimal(modele, X_train, Y_train)
    resultats[nom] = (modele, seuil, f1_cv)
    print(f"{nom} : seuil = {seuil:.3f} | F1 validation croisée (train) = {f1_cv:.4f}")

nom_retenu = max(resultats, key=lambda k: resultats[k][2])
modele_retenu, seuil_final, _ = resultats[nom_retenu]
print(f"\nModèle retenu : {nom_retenu}")

Random Forest : seuil = 0.934 | F1 validation croisée (train) = 0.7393
Gradient boosting : seuil = 0.942 | F1 validation croisée (train) = 0.7663

Modèle retenu : Gradient boosting


In [14]:
modele_retenu.fit(X_train, Y_train)

proba_test = modele_retenu.predict_proba(X_test)[:, 1]
y_pred_test = (proba_test >= seuil_final).astype(int)

print(f"--- Test set : {nom_retenu}, seuil = {seuil_final:.3f} ---")
print("Matrice de confusion :\n", confusion_matrix(Y_test, y_pred_test))
print("\nRapport de classification :\n", classification_report(Y_test, y_pred_test, digits=3))
print(f"F1 test : {f1_score(Y_test, y_pred_test):.4f}")

--- Test set : Gradient boosting, seuil = 0.942 ---
Matrice de confusion :
 [[54783   297]
 [  491  1345]]

Rapport de classification :
               precision    recall  f1-score   support

           0      0.991     0.995     0.993     55080
           1      0.819     0.733     0.773      1836

    accuracy                          0.986     56916
   macro avg      0.905     0.864     0.883     56916
weighted avg      0.986     0.986     0.986     56916

F1 test : 0.7734


In [15]:
y_pred_train = (modele_retenu.predict_proba(X_train)[:, 1] >= seuil_final).astype(int)
f1_train = f1_score(Y_train, y_pred_train)
f1_test = f1_score(Y_test, y_pred_test)

print(f"F1 train : {f1_train:.4f} | F1 test : {f1_test:.4f} | écart : {f1_train - f1_test:.4f}")

F1 train : 0.7680 | F1 test : 0.7734 | écart : -0.0055


# Lecture du résultat

- **Modèle retenu** : gradient boosting, seuil de décision 0,942.
- **Test set** : F1 = 0,773, rappel = 73,3 %, précision = 81,9 %.
- **Écart train/test** : F1 train 0,768 contre 0,773 en test, soit −0,006. Pas de surapprentissage visible.

# Partie 3 : Prédictions et fichier de soumission

Le modèle retenu est réentraîné sur les 284 580 lignes labellisées, puis appliqué au fichier test. Le seuil de décision reste celui déterminé en validation croisée.

In [16]:
modele_soumission = clone(modele_retenu)
modele_soumission.fit(X, Y)
print("Modèle réentraîné sur", X.shape[0], "lignes labellisées.")

data_test = pd.read_csv('conversion_data_test.csv')
print("Données test (sans labels) :", data_test.shape)

proba_soum = modele_soumission.predict_proba(data_test[features_list])[:, 1]
predictions = (proba_soum >= seuil_final).astype(int)

print(f"Part de prédictions à 1 : {predictions.mean():.2%} "
      f"(taux observé dans le train : {Y.mean():.2%})")

pd.DataFrame({'converted': predictions}).to_csv(
    'conversion_data_test_predictions_antoine.csv', index=False)
print("Fichier créé : conversion_data_test_predictions_antoine.csv")

Modèle réentraîné sur 284580 lignes labellisées.
Données test (sans labels) : (31620, 5)
Part de prédictions à 1 : 2.97% (taux observé dans le train : 3.23%)
Fichier créé : conversion_data_test_predictions_antoine.csv


# Partie 4 : Facteurs explicatifs

L'importance par permutation mesure la baisse de F1 lorsqu'on mélange au hasard une variable, les autres restant inchangées. Contrairement à l'importance par impureté du Random Forest, elle n'est pas biaisée vers les variables à forte cardinalité. Elle est calculée sur le test set, qui n'a pas servi à choisir le modèle.

In [17]:
def f1_avec_seuil(estimator, X_, y_):
    return f1_score(y_, (estimator.predict_proba(X_)[:, 1] >= seuil_final).astype(int))

perm = permutation_importance(modele_retenu, X_test, Y_test, scoring=f1_avec_seuil,
                              n_repeats=5, random_state=42)

imp_df = pd.DataFrame({
    'feature': features_list,
    'importance': perm.importances_mean,
    'std': perm.importances_std
}).sort_values('importance', ascending=False)

display(imp_df)

fig_imp = px.bar(imp_df, x='importance', y='feature', orientation='h', error_x='std',
                 title="Importance des variables (baisse de F1 après permutation, test set)")
fig_imp.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_imp.show()

,feature,importance,std
4,total_pages_visited,0.732646,0.003836
2,new_user,0.061498,0.003155
0,country,0.040899,0.005360
1,age,0.021912,0.003577
3,source,0.000486,0.000531


# Lecture des facteurs

- **total_pages_visited** : baisse de F1 de 0,733 (± 0,004). Variable dominante, loin devant les autres.
- **new_user** : baisse de F1 de 0,061 (± 0,003).
- **country** : baisse de F1 de 0,041 (± 0,005). Plus important que l'âge.
- **age** : baisse de F1 de 0,022 (± 0,004).
- **source** : baisse de F1 de 0,0005, négligeable.

# Recommandations

Le modèle prédit une conversion, mais il ne montre pas qu'une action la provoque. Chaque piste est donc une hypothèse à valider par un test A/B.

1. **Navigation** (`total_pages_visited`) : les visiteurs qui consultent davantage de pages convertissent nettement plus. Tester des liens internes et des contenus recommandés, puis vérifier si la conversion suit.
2. **Nouveaux visiteurs** (`new_user`) : tester un message d'accueil incitant à l'inscription dès la première visite.
3. **Âge** (`age`) : la tranche 10-20 ans convertit le plus, ce qui peut refléter un problème de données (mineurs, âges aberrants). À vérifier avant toute personnalisation.
4. **Pays** (`country`) : écart important (importance 0,041), mais à vérifier avant toute action (effectifs par pays, qualité du tracking). **Source** : importance négligeable. Le modèle ne mesure pas l'effet d'un investissement publicitaire, donc une réallocation de budget demande un test séparé.

# Conclusion

**Objectif** : prédire les inscriptions à la newsletter à partir de quelques informations de visite, avec le F1 comme critère.

**Démarche** : baseline (régression logistique, Random Forest), optimisation sur échantillon, sélection entre Random Forest et gradient boosting par validation croisée sur le train (seuil optimisé), évaluation unique sur le test.

**Résultat** : gradient boosting, seuil 0,942, F1 test = 0,773, rappel = 73,3 %, précision = 81,9 %. La baseline logistique obtenait 0,512.

**Facteurs explicatifs** : le nombre de pages visitées domine très nettement, suivi du statut de nouveau visiteur, puis du pays. L'âge a un effet plus faible, et la source est négligeable.

**Limites**
- Les données sont observationnelles : elles montrent des associations, pas des effets causaux.
- Écart train/test de 0,006 : faible.
- Le seuil maximise le F1. Un autre compromis serait préférable si un faux positif (relance inutile) ou un faux négatif (prospect perdu) coûte plus cher.
- Âges aberrants (jusqu'à 123) et tranche 10-20 ans à vérifier. Écarts entre pays non expliqués à ce stade.
- Le seuil est choisi sur la validation croisée du train, donc le F1 de validation est légèrement optimiste. Le F1 test (0,773), obtenu sur des données jamais utilisées pour le choix, est plus fiable.

**Suite** : tester les recommandations en A/B pour mesurer leur effet réel.